# [Introduction to Data Science](http://datascience-intro.github.io/1MS041-2026/)    
## 1MS041, 2026 
&copy;2026 Raazesh Sainudiin, Benny Avelin. [Attribution 4.0 International     (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/)

# Random Variables

This notebook accompanies Chapter 2, Sections 2.1--2.5. We distinguish a
random variable from code that samples its distribution, compute discrete
and continuous laws, transform random variables, and calculate expectations
and variances.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(202)


## Random variables turn outcomes into numbers

In the model of two independent fair coin tosses, let
$\Omega=\{HH,HT,TH,TT\}$, with probability $1/4$ at every outcome.
Define $X:\Omega\to\mathbb R$ to be the number of Heads. The function
$X$ is the random variable. A Python call that returns simulated values
with the same distribution is a **sampler**, not the random variable itself.


In [ ]:
outcomes = np.array(["HH", "HT", "TH", "TT"])
outcome_probabilities = np.full(4, 0.25)
x_of_outcome = np.array([2, 1, 1, 0])

x_values = np.array([0, 1, 2])
x_pmf = np.array([
    outcome_probabilities[x_of_outcome == value].sum()
    for value in x_values
])
x_cdf = np.cumsum(x_pmf)

for value, mass, cumulative in zip(x_values, x_pmf, x_cdf):
    print(f"x={value}: P(X=x)={mass:.2f}, F_X(x)={cumulative:.2f}")


For a discrete random variable, the probability mass function (PMF) is
$p_X(x)=\mathbb P(X=x)$. The distribution function (DF or CDF) is
$F_X(t)=\mathbb P(X\leq t)$. It is defined for every real $t$, is
nondecreasing and right-continuous, and has limits $0$ and $1$ at the
two ends of the real line.


In [ ]:
grid = np.linspace(-0.5, 2.5, 301)
cdf_on_grid = sum(
    mass * (grid >= value) for value, mass in zip(x_values, x_pmf)
)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.step(grid, cdf_on_grid, where="post")
ax.set(xlabel="t", ylabel="$F_X(t)$", ylim=(-0.05, 1.05), title="CDF of the number of Heads")
plt.show()


## Working with continuous distributions

A random variable is called continuous here when its distribution has a
density $f$: $f(x)\geq0$, its integral over $\mathbb R$ is $1$,
and probabilities are integrals of $f$. Merely taking values in an
uncountable set is not enough.

For $X\sim\mathrm{Uniform}([a,b])$,

$$
f_X(x)=\frac{1}{b-a}\mathbf 1_{[a,b]}(x).
$$

For $T\sim\mathrm{Exponential}(\lambda)$, where $\lambda>0$ is a rate,
$f_T(t)=\lambda e^{-\lambda t}$ for $t\geq0$ and
$F_T(t)=1-e^{-\lambda t}$ there.


In [ ]:
a, b = -2.0, 3.0
uniform_sample = rng.uniform(a, b, size=20_000)
grid = np.linspace(a - 0.5, b + 0.5, 400)
uniform_density = np.where((grid >= a) & (grid <= b), 1 / (b - a), 0.0)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(uniform_sample, bins=40, density=True, alpha=0.55, label="simulation")
ax.plot(grid, uniform_density, color="black", label="density $1/(b-a)$")
ax.set(xlabel="x", ylabel="density")
ax.legend()
plt.show()


## Transforming a random variable

For a discrete transformation $Y=g(X)$, add the masses of all values of
$X$ with the same image. For a fair die $X$, let $Y=(X-3)^2$.
This is a many-to-one transformation.


In [ ]:
die_values = np.arange(1, 7)
transformed_values = (die_values - 3) ** 2

y_pmf = {}
for value in transformed_values:
    y_pmf[value] = y_pmf.get(value, 0.0) + 1 / 6

print("Y values and probabilities:")
for value, mass in sorted(y_pmf.items()):
    print(f"P(Y={value}) = {mass:.3f}")
print("Mass sums to", sum(y_pmf.values()))


For a continuous example, let $X\sim\mathrm{Uniform}([-1,1])$ and
$Y=X^2$. For $0<y<1$, both roots $\sqrt y$ and $-\sqrt y$
contribute, giving

$$
f_Y(y)=\frac{1}{2\sqrt y},\qquad 0<y<1.
$$

The density is unbounded near zero but still integrates to one.


In [ ]:
x_sample = rng.uniform(-1, 1, size=50_000)
y_sample = x_sample**2
y_grid = np.linspace(0.002, 1, 500)
y_density = 1 / (2 * np.sqrt(y_grid))

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(y_sample, bins=60, density=True, alpha=0.55, label="simulation of $X^2$")
ax.plot(y_grid, y_density, color="black", label=r"$1/(2\sqrt{y})$")
ax.set(xlabel="y", ylabel="density", ylim=(0, 5))
ax.legend()
plt.show()


## Mean and variation

When the relevant sums or integrals are finite,
$\mathbb E[X]$ is the probability-weighted mean and

$$
\operatorname{Var}(X)=\mathbb E[(X-\mathbb E[X])^2]
=\mathbb E[X^2]-\mathbb E[X]^2.
$$

Variance is the second **central** moment; $\mathbb E[X^2]$ is the second
raw moment and is generally different.


In [ ]:
mean_x = np.sum(x_values * x_pmf)
second_moment_x = np.sum(x_values**2 * x_pmf)
variance_x = second_moment_x - mean_x**2

print(f"E[X] = {mean_x:.3f}")
print(f"E[X^2] = {second_moment_x:.3f}")
print(f"Var(X) = {variance_x:.3f}")


## Try it yourself

1. Let $X$ be a fair die and $Y=\mathbf 1_{\{X\geq5\}}$. Find the PMF,
   expectation, and variance of $Y$.
2. If $T\sim\mathrm{Exponential}(2)$, compute
   $\mathbb P(1<T\leq2)$ from its CDF.
3. Derive the CDF of $X^2$ when $X\sim\mathrm{Uniform}([-1,1])$, then
   differentiate it on $(0,1)$ to recover the density above.
